# DWBI Week 5 — The ETL Process

**Ali Raza** · `su92-bsdsm-f23-018` · **BSDS-7A** · Superior University

PakFreight's Week-2 merge rebuilt as a pipeline that reruns: 37 rows land, 35 append, 34 load — 578,650 kg and PKR 4,808,000, exactly. Every number below is computed from `data/source/`, `data/dims/` or `data/night2/` in the notebook itself; nothing is typed in as a constant except the assertion target it is checked against.

## T6 — Loud or quiet (12 marks)

Six failures — four scheduling accidents and two defects. The classification
test is the same as T1's: **loud** means the run itself stops and somebody is
told; **quiet** means the load succeeds and something published is wrong. A
pipeline that never fails is not reliable. It is unmonitored.

In [1]:

from pathlib import Path
import pandas as pd

BASE = Path("data"); SRC = BASE / "source"; DIMS = BASE / "dims"
CONTRACT = ["ShipmentID", "ShipDate", "OriginRaw", "DestRaw",
            "WeightKG", "FreightCostPKR", "StatusRaw"]
RENAME = {
    "marine": {"Shipment_ID": "ShipmentID", "Ship_Date": "ShipDate",
               "Origin_Port": "OriginRaw", "Dest_Port": "DestRaw",
               "Weight_KG": "WeightKG", "Cost_PKR": "FreightCostPKR",
               "Status": "StatusRaw"},
    "road": {"trip_id": "ShipmentID", "trip_date": "ShipDate",
             "from_city": "OriginRaw", "to_city": "DestRaw",
             "weight_kg": "WeightKG", "cost_pkr": "FreightCostPKR",
             "status": "StatusRaw"},
    "rail": {"Consignment No": "ShipmentID", "Dispatch Date": "ShipDate",
             "From Station": "OriginRaw", "To Station": "DestRaw",
             "Weight (Tons)": "WeightTons",
             "Freight Charges (PKR)": "FreightCostPKR", "Status": "StatusRaw"},
}
MODE = {"marine": 1, "rail": 2, "road": 3}

def extract(src, name):
    if name == "rail":
        return pd.read_excel(src / "rail_shipments.xlsx", sheet_name="Rail", skiprows=2)
    return pd.read_csv(src / f"{name}_shipments.csv")

def transform(df, name):
    df = df.rename(columns=RENAME[name])
    ids = df.ShipmentID.astype("string").str.strip()
    df = df[ids.notna() & (ids != "") & (ids.str.upper() != "TOTAL")].copy()
    for c in ("OriginRaw", "DestRaw"):
        df[c] = df[c].astype("string").str.strip()
    df["StatusRaw"] = (df.StatusRaw.astype("string")
                       .str.replace(r"\s+", " ", regex=True)
                       .str.strip().str.title())
    if name == "rail":
        df["ShipmentID"] = "RL-" + df.ShipmentID.astype(int).astype(str)
        df["WeightKG"] = df.WeightTons * 1000
        df = df.drop(columns=["WeightTons"])
        df["ShipDate"] = pd.to_datetime(df.ShipDate)
    else:
        df["ShipmentID"] = df.ShipmentID.astype("string").str.strip().str.upper()
        fmt = "%d/%m/%Y" if name == "marine" else "%Y-%m-%d"
        df["ShipDate"] = pd.to_datetime(df.ShipDate, format=fmt)
    df["WeightKG"] = df.WeightKG.astype("int64")
    df["FreightCostPKR"] = df.FreightCostPKR.astype("int64")
    df["ModeKey"] = MODE[name]
    return df[CONTRACT + ["ModeKey"]]

def alias_lookup():
    dim = pd.read_csv(DIMS / "DimCity.csv")
    out = (dim[["CityKey", "RawAliases"]]
           .assign(RawAliases=lambda d: d.RawAliases.str.split(" | ", regex=False))
           .explode("RawAliases"))
    out["RawAliases"] = out.RawAliases.str.strip()
    out = out[out.RawAliases.notna() & (out.RawAliases != "")]
    return out.rename(columns={"RawAliases": "RawSpelling"})

def pipeline(src=SRC):
    parts = [transform(extract(src, nm), nm) for nm in ("marine", "rail", "road")]
    fact = pd.concat(parts, ignore_index=True)
    appended = len(fact)
    fact = fact.drop_duplicates(subset=["ShipmentID"])
    alias = alias_lookup()
    for raw, key in (("OriginRaw", "OriginCityKey"), ("DestRaw", "DestCityKey")):
        fact = fact.merge(alias.rename(columns={"RawSpelling": raw, "CityKey": key}),
                          on=raw, how="left")
        fact[key] = fact[key].fillna(-1).astype("int64")
    return fact.reset_index(drop=True), appended, parts

fact, appended, parts = pipeline()
# Demonstrate three of the six failures on the real pipeline output
# (a) status never trimmed/cased: raw distinct count vs conformed count
marine_raw = pd.read_csv("data/source/marine_shipments.csv", dtype=str, keep_default_na=False)
road_raw = pd.read_csv("data/source/road_shipments.csv", dtype=str, keep_default_na=False)
rail_raw = pd.read_excel("data/source/rail_shipments.xlsx", sheet_name="Rail", skiprows=2, dtype=str)
road_real = road_raw[~road_raw.trip_id.isin(["", "TOTAL"])]
raw_statuses = set(marine_raw.Status) | set(rail_raw.Status) | set(road_real.status)
print(f"raw distinct statuses: {len(raw_statuses)} -> conformed: {fact.StatusRaw.nunique()} (check 3)")
# (b) double-fired run with a bare insert: append the fact to itself
doubled = pd.concat([fact, fact], ignore_index=True)
print(f"double-fired: {len(doubled)} rows, {doubled.WeightKG.sum():,} kg (checks 5, 7, 8 all fail)")
# (c) a later step naming a renamed column: the loud failure, reproduced
try:
    _ = fact["Status"]
except KeyError as e:
    print(f"renamed-column error (loud, run stops): KeyError {e}")
assert fact.StatusRaw.nunique() == 3 and len(doubled) == 68 and doubled.WeightKG.sum() == 2 * 578650

raw distinct statuses: 17 -> conformed: 3 (check 3)
double-fired: 68 rows, 1,157,300 kg (checks 5, 7, 8 all fail)
renamed-column error (loud, run stops): KeyError 'Status'


**Analysis.** Three failures are demonstrated rather than asserted: untrimmed status really does offer 17 choices against 3 after conforming, a double-fired bare insert really does double the table to 68 rows and 1,157,300 kg (so checks 5, 7 and 8 all fail at once), and a step naming the pre-rename column really does raise a KeyError and stop the run. The other three failures are scheduling accidents with no data to run, so they are classified by argument in the table below.

| # | The accident, or the defect | Loud or quiet | Which check catches it | What makes it safe |
|---|---|---|---|---|
| 1 | The nightly run never fired on Tuesday | Quiet | **None of the eight** — and that is the point | A schedule monitor and a freshness stamp on the 08:00 report ("data as of …"). The eight checks are *data* checks: on yesterday's unchanged warehouse they all still pass, so a run that never happens is invisible to them. The safety is outside the pipeline: alert when no run log appears, and never publish a report without its as-of date. |
| 2 | The same run fired twice, loading with a bare insert | **Both** | Checks 5, 7, 8 (if it loads); the run error (if it cannot) | An **idempotent load**: a key constraint or a truncate-and-reload/merge on ShipmentID, so the second run either replaces the first or fails loudly instead of appending a second copy. Bare insert is the unsafe part — it makes "ran twice" indistinguishable from "shipped twice". |
| 3 | The file was still being written when extract read it | Quiet | Checks 5, 7, 8 | A file-ready signal before extract runs (a marker file, a stable file size over two polls, or a row-count handshake against the sender's control total). Extract must never read a file that is still open for writing; the partial read otherwise loads cleanly, short by however many rows had not yet been written. |
| 4 | Change Type was used instead of Using Locale on the marine dates | Quiet | **Check 4** | Change Type **Using Locale, English (United Kingdom)** — or in pandas, `format="%d/%m/%Y"` stated explicitly. Checks 7 and 8 pass regardless (see T5), so only the month-set check catches it. |
| 5 | The status column was never trimmed or cased | Quiet | **Check 3** | Trim, then Clean, then Proper Case, as a named step before any grouping or lookup. The check reads 17 instead of 3; the fix is one step, and it has to sit before the step that consumes the values, not after. |
| 6 | A later step still names a column an earlier step renamed | **Loud** | The run itself — no check is needed | Named steps and reading the step the error names. This is the one failure you have no work to do about: the run stops, the warehouse is unchanged, and the error message points at the exact step. It is doing you a favour. |

Self-check, computed from the table: 1 loud (row 6), 4 quiet (rows 1, 3, 4, 5),
1 both (row 2).

### The answer that is not the obvious one (4 marks)

Everybody says they would rather have the loud failure — a red job, a page, a stale report
with a stale-as-of stamp — than the quiet one, where the dashboard is green and the numbers are
wrong until a quarter closes. What follows from that preference is a statement about where
engineering effort in a pipeline actually goes: **it goes into making failures loud, not into
making failures rare.** You cannot make a pipeline that never meets a half-written file, a
missed schedule or a renamed column, but you can decide, for each of those, whether the pipeline
tells somebody or publishes anyway. So the effort goes into the unglamorous machinery that
converts quiet failures into loud ones — assertions on the eight checks that fail the run
instead of warning about it, uniqueness constraints that turn a double load into an error,
freshness alerts that turn a missed run into a page, control totals that turn a partial read
into a rejected file, and a −1 count that is printed and written down rather than absorbed.
A pipeline that never fails has not achieved reliability; it has achieved silence, and silence
is the most expensive failure mode there is, because it is discovered by the business instead
of by the pipeline.